# JEV RAG — FastAPI Documentation

A compact, reproducible notebook for building and inspecting a retrieval-augmented generation (RAG) system.

**Current scope:** document ingestion, BGE embeddings, Neon PostgreSQL/pgvector storage, and dense/BM25/hybrid retrieval.

**Evaluation note:** retrieval relevance labels are still provisional. Do not treat metric scores as final until the question–chunk relevance judgments are reviewed.


## 1. Project setup and configuration
Run this section once per fresh Colab runtime. Keep `.env` private; never print secrets.

In [1]:
import os

PROJECT_DIR = "/content/jev-rag"

os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)

print("Project directory:", os.getcwd())

Project directory: /content/jev-rag


In [2]:
folders = [
    "data/raw",
    "data/processed",
    "src/ingestion",
    "src/retrieval",
    "src/evaluation",
    "src/graph",
    "src/generation",
    "tests",
    "notebooks",
    "logs"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully!")

Project folders created successfully!


In [3]:
requirements = """
langgraph
langchain-core
langchain-text-splitters
langchain-openai
openai
sentence-transformers
psycopg[binary]
pgvector
rank-bm25
ragas
datasets
pandas
numpy
scikit-learn
pydantic
python-dotenv
requests
beautifulsoup4
trafilatura
"""

with open("requirements.txt", "w") as f:
    f.write(requirements.strip() + "\n")

print("requirements.txt created!")

requirements.txt created!


In [4]:
%pip install -r /content/jev-rag/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.6/145.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.7/158.7 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.7/254.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.4/269.4 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
from pathlib import Path
from getpass import getpass

PROJECT_DIR = Path("/content/jev-rag")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

env_path = PROJECT_DIR / ".env"

openrouter_key = getpass("Paste your OpenRouter API key: ").strip()
database_url = getpass("Paste your Neon PostgreSQL connection string: ").strip()

if not openrouter_key:
    raise ValueError("OpenRouter API key cannot be empty.")

if not database_url.startswith(("postgresql://", "postgres://")):
    raise ValueError("Please enter a valid PostgreSQL connection string.")

# Preserve existing variables, updating only the two specified keys.
existing = {}
if env_path.exists():
    for line in env_path.read_text().splitlines():
        if "=" in line and not line.lstrip().startswith("#"):
            key, value = line.split("=", 1)
            existing[key.strip()] = value

existing["OPENROUTER_API_KEY"] = openrouter_key
existing["DATABASE_URL"] = database_url

env_path.write_text(
    "".join(f"{key}={value}\n" for key, value in existing.items())
)
env_path.chmod(0o600)

print("Environment file saved:", env_path)
print("API key and database URL were not displayed.")

Paste your OpenRouter API key: ··········
Paste your Neon PostgreSQL connection string: ··········
Environment file saved: /content/jev-rag/.env
API key and database URL were not displayed.


## 2. Database setup
Create the pgvector extension, table, and indexes. These are safe to rerun because the SQL uses `IF NOT EXISTS`.

In [6]:
import os
import psycopg
from dotenv import load_dotenv

load_dotenv("/content/jev-rag/.env", override=True)

DATABASE_URL = os.getenv("DATABASE_URL")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

assert DATABASE_URL, "DATABASE_URL is missing from .env"
assert OPENROUTER_API_KEY, "OPENROUTER_API_KEY is missing from .env"

with psycopg.connect(DATABASE_URL, connect_timeout=20) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version();")
        print("PostgreSQL connection successful.")
        print("Database version:", cur.fetchone()[0].split(",")[0])

        cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
        cur.execute(
            "SELECT extversion FROM pg_extension WHERE extname = 'vector';"
        )
        result = cur.fetchone()

        if result:
            print("pgvector enabled. Version:", result[0])
        else:
            print("pgvector extension was not found.")

print("Environment variables loaded successfully.")

PostgreSQL connection successful.
Database version: PostgreSQL 18.6 (c021049) on aarch64-unknown-linux-gnu
pgvector enabled. Version: 0.8.6
Environment variables loaded successfully.


In [35]:
import os
from getpass import getpass

os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API key: ")

Enter your OpenRouter API key: ··········


In [7]:
from pgvector.psycopg import register_vector

with psycopg.connect(DATABASE_URL, connect_timeout=20) as conn:
    register_vector(conn)

    with conn.cursor() as cur:
        cur.execute("SELECT '[1,2,3]'::vector <=> '[1,2,4]'::vector;")
        distance = cur.fetchone()[0]

        print("Vector operation successful.")
        print("Cosine distance:", distance)

Vector operation successful.
Cosine distance: 0.00853986601633272


In [8]:
gitignore_path = Path("/content/jev-rag/.gitignore")

existing_text = (
    gitignore_path.read_text()
    if gitignore_path.exists()
    else ""
)

required_rules = [".env", ".env.*", "!.env.example"]
lines = existing_text.splitlines()

for rule in required_rules:
    if rule not in lines:
        lines.append(rule)

gitignore_path.write_text("\n".join(lines) + "\n")
print(".gitignore updated.")

.gitignore updated.


In [9]:
import os
import psycopg
from dotenv import load_dotenv

load_dotenv("/content/jev-rag/.env", override=True)
DATABASE_URL = os.environ["DATABASE_URL"]

create_table_sql = """
CREATE TABLE IF NOT EXISTS document_chunks (
    chunk_id BIGSERIAL PRIMARY KEY,
    document_id TEXT NOT NULL,
    title TEXT NOT NULL,
    section TEXT,
    source_url TEXT NOT NULL,
    content TEXT NOT NULL,
    content_hash TEXT NOT NULL,
    metadata JSONB NOT NULL DEFAULT '{}'::jsonb,
    embedding VECTOR(384),
    search_vector TSVECTOR GENERATED ALWAYS AS (
        to_tsvector('english', coalesce(title, '') || ' ' ||
                               coalesce(section, '') || ' ' ||
                               coalesce(content, ''))
    ) STORED,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    UNIQUE (document_id, content_hash)
);

CREATE INDEX IF NOT EXISTS idx_chunks_embedding
    ON document_chunks USING hnsw (embedding vector_cosine_ops);

CREATE INDEX IF NOT EXISTS idx_chunks_search
    ON document_chunks USING gin (search_vector);

CREATE INDEX IF NOT EXISTS idx_chunks_document_id
    ON document_chunks (document_id);
"""

with psycopg.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
        cur.execute(create_table_sql)

        cur.execute("""
            SELECT column_name, data_type, udt_name
            FROM information_schema.columns
            WHERE table_name = 'document_chunks'
            ORDER BY ordinal_position;
        """)
        columns = cur.fetchall()

print("Table and indexes created successfully.")
for column in columns:
    print(column)

Table and indexes created successfully.
('chunk_id', 'bigint', 'int8')
('document_id', 'text', 'text')
('title', 'text', 'text')
('section', 'text', 'text')
('source_url', 'text', 'text')
('content', 'text', 'text')
('content_hash', 'text', 'text')
('metadata', 'jsonb', 'jsonb')
('embedding', 'USER-DEFINED', 'vector')
('search_vector', 'tsvector', 'tsvector')
('created_at', 'timestamp with time zone', 'timestamptz')


## 3. Download and clean documentation
The source set is six official FastAPI pages.

In [10]:
import requests
import trafilatura
from pathlib import Path

RAW_DIR = Path("/content/jev-rag/data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

DOCS = [
    "https://fastapi.tiangolo.com/",
    "https://fastapi.tiangolo.com/tutorial/",
    "https://fastapi.tiangolo.com/tutorial/first-steps/",
    "https://fastapi.tiangolo.com/tutorial/path-params/",
    "https://fastapi.tiangolo.com/tutorial/query-params/",
    "https://fastapi.tiangolo.com/tutorial/body/",
]

documents = []
session = requests.Session()
session.headers.update({
    "User-Agent": "JEV-RAG-Learning-Project/1.0"
})

for url in DOCS:
    try:
        response = session.get(url, timeout=30)
        response.raise_for_status()

        text = trafilatura.extract(
            response.text,
            include_links=True,
            include_tables=True,
        )

        if not text or len(text.strip()) < 100:
            print(f"Skipped (no useful text): {url}")
            continue

        title = url.rstrip("/").split("/")[-1] or "FastAPI Documentation"

        documents.append({
            "title": title,
            "source_url": url,
            "content": text.strip(),
        })

        filename = title.replace("-", "_") + ".txt"
        (RAW_DIR / filename).write_text(text.strip(), encoding="utf-8")

        print(f"Downloaded: {url} | {len(text)} characters")

    except requests.RequestException as exc:
        print(f"Failed: {url} | {type(exc).__name__}: {exc}")

print(f"\nSuccessfully parsed {len(documents)} of {len(DOCS)} pages.")

Downloaded: https://fastapi.tiangolo.com/ | 16674 characters
Downloaded: https://fastapi.tiangolo.com/tutorial/ | 7107 characters
Downloaded: https://fastapi.tiangolo.com/tutorial/first-steps/ | 13413 characters
Downloaded: https://fastapi.tiangolo.com/tutorial/path-params/ | 10980 characters
Downloaded: https://fastapi.tiangolo.com/tutorial/query-params/ | 5566 characters
Downloaded: https://fastapi.tiangolo.com/tutorial/body/ | 7710 characters

Successfully parsed 6 of 6 pages.


In [11]:
assert documents, "No documents were extracted. Check the download errors."

for doc in documents[:2]:
    print("=" * 80)
    print("Title:", doc["title"])
    print("URL:", doc["source_url"])
    print("Content preview:")
    print(doc["content"][:700])
    print()

Title: fastapi.tiangolo.com
URL: https://fastapi.tiangolo.com/
Content preview:
[FastAPI Conf '26 October 28, 2026 Amsterdam, NL All about FastAPI, right from the source. Learn more](https://fastapiconf.com)
FastAPI[¶](#fastapi)
FastAPI framework, high performance, easy to learn, fast to code, ready for production
Documentation: [https://fastapi.tiangolo.com](https://fastapi.tiangolo.com)
Source Code: [https://github.com/fastapi/fastapi](https://github.com/fastapi/fastapi)
FastAPI is a modern, fast (high-performance), web framework for building APIs with Python based on standard Python type hints.
The key features are:
- Fast: Very high performance, on par with NodeJS and Go (thanks to Starlette and Pydantic). [One of the fastest Python frameworks available](#performanc

Title: tutorial
URL: https://fastapi.tiangolo.com/tutorial/
Content preview:
Tutorial - User Guide[¶](#tutorial-user-guide)
This tutorial shows you how to use FastAPI with most of its features, step by step.
Each secti

In [12]:
import re
import html
from bs4 import BeautifulSoup

def clean_document_text(text: str) -> str:
    # Decode HTML entities
    text = html.unescape(text)

    # Remove Markdown link destinations, retaining readable link text
    text = re.sub(r'\[([^\]]+)\]\((?:\\.|[^)])*\)', r'\1', text)

    # Remove leftover Markdown URL wrappers and escape characters
    text = re.sub(r'\\([()\[\]])', r'\1', text)
    text = re.sub(r'https?://\S+', '', text)

    # Remove common HTML tags if any remain
    text = BeautifulSoup(text, "html.parser").get_text(" ", strip=True)

    # Normalize whitespace
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Remove blank or very short lines
    lines = [line.strip() for line in text.splitlines() if len(line.strip()) > 1]
    return '\n'.join(lines).strip()

for doc in documents:
    doc["content"] = clean_document_text(doc["content"])

print(f"Cleaned {len(documents)} documents.")

for doc in documents[:2]:
    print("\n" + "=" * 70)
    print("Title:", doc["title"])
    print("URL:", doc["source_url"])
    print(doc["content"][:600])

Cleaned 6 documents.

Title: fastapi.tiangolo.com
URL: https://fastapi.tiangolo.com/
FastAPI Conf '26 October 28, 2026 Amsterdam, NL All about FastAPI, right from the source. Learn more
FastAPI¶
FastAPI framework, high performance, easy to learn, fast to code, ready for production
Documentation:
Source Code:
FastAPI is a modern, fast (high-performance), web framework for building APIs with Python based on standard Python type hints.
The key features are:
- Fast: Very high performance, on par with NodeJS and Go (thanks to Starlette and Pydantic). One of the fastest Python frameworks available.
- Fast to code: Increase the speed to develop features by about 200% to 300%. *
- Few

Title: tutorial
URL: https://fastapi.tiangolo.com/tutorial/
Tutorial - User Guide¶
This tutorial shows you how to use FastAPI with most of its features, step by step.
Each section gradually builds on the previous ones, but it's structured to separate topics, so that you can go directly to any specific one to sol

In [13]:
for i, doc in enumerate(documents, start=1):
    print(
        f"{i}. {doc['title']} | "
        f"{len(doc['content'])} characters | "
        f"URL: {doc['source_url']}"
    )

assert all(len(doc["content"]) > 100 for doc in documents), \
    "At least one document has too little text. Inspect it before continuing."

print("\nQuality check passed.")

1. fastapi.tiangolo.com | 13206 characters | URL: https://fastapi.tiangolo.com/
2. tutorial | 4818 characters | URL: https://fastapi.tiangolo.com/tutorial/
3. first-steps | 9830 characters | URL: https://fastapi.tiangolo.com/tutorial/first-steps/
4. path-params | 9679 characters | URL: https://fastapi.tiangolo.com/tutorial/path-params/
5. query-params | 4678 characters | URL: https://fastapi.tiangolo.com/tutorial/query-params/
6. body | 6974 characters | URL: https://fastapi.tiangolo.com/tutorial/body/

Quality check passed.


## 4. Chunk documents
Chunk size is 800 characters with 120-character overlap.

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import hashlib

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""],
    add_start_index=True,
)

chunk_records = []

for doc in documents:
    chunks = text_splitter.create_documents(
        texts=[doc["content"]],
        metadatas=[{
            "document_id": doc["source_url"],
            "title": doc["title"],
            "source_url": doc["source_url"],
        }],
    )

    for index, item in enumerate(chunks):
        content = item.page_content.strip()

        if not content:
            continue

        chunk_records.append({
            "document_id": doc["source_url"],
            "title": doc["title"],
            "section": f"chunk_{index + 1}",
            "source_url": doc["source_url"],
            "content": content,
            "content_hash": hashlib.sha256(
                content.encode("utf-8")
            ).hexdigest(),
            "metadata": {
                **item.metadata,
                "chunk_index": index,
                "start_index": item.metadata.get("start_index"),
            },
        })

print("Documents:", len(documents))
print("Total chunks:", len(chunk_records))
print(
    "Average chunk length:",
    round(sum(len(c["content"]) for c in chunk_records) / len(chunk_records))
)

assert chunk_records, "No chunks were generated."

Documents: 6
Total chunks: 75
Average chunk length: 724


In [15]:
for i, chunk in enumerate(chunk_records[:3], start=1):
    print(f"\n--- Chunk {i} ({len(chunk['content'])} characters) ---")
    print("Title:", chunk["title"])
    print("URL:", chunk["source_url"])
    print(chunk["content"][:500])


--- Chunk 1 (743 characters) ---
Title: fastapi.tiangolo.com
URL: https://fastapi.tiangolo.com/
FastAPI Conf '26 October 28, 2026 Amsterdam, NL All about FastAPI, right from the source. Learn more
FastAPI¶
FastAPI framework, high performance, easy to learn, fast to code, ready for production
Documentation:
Source Code:
FastAPI is a modern, fast (high-performance), web framework for building APIs with Python based on standard Python type hints.
The key features are:
- Fast: Very high performance, on par with NodeJS and Go (thanks to Starlette and Pydantic). One of the fastest Python framewo

--- Chunk 2 (641 characters) ---
Title: fastapi.tiangolo.com
URL: https://fastapi.tiangolo.com/
- Intuitive: Great editor support. Completion everywhere. Less time debugging.
- Easy: Designed to be easy to use and learn. Less time reading docs.
- Short: Minimize code duplication. Multiple features from each parameter declaration. Fewer bugs.
- Robust: Get production-ready code. With automatic inter

In [16]:
lengths = [len(chunk["content"]) for chunk in chunk_records]

print("Total chunks:", len(lengths))
print("Average:", round(sum(lengths) / len(lengths)))
print("Minimum:", min(lengths))
print("Maximum:", max(lengths))

Total chunks: 75
Average: 724
Minimum: 177
Maximum: 800


## 5. Generate embeddings and store chunks
The embedding model produces 384-dimensional vectors. Existing identical document chunks are skipped by the database uniqueness constraint.

In [17]:
from sentence_transformers import SentenceTransformer
import numpy as np

try:
    embedding_model
except NameError:
    embedding_model = SentenceTransformer(
        "BAAI/bge-small-en-v1.5",
        device="cpu",
    )

texts = [chunk["content"] for chunk in chunk_records]

embeddings = embedding_model.encode(
    texts,
    batch_size=16,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print("Embedding shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)

assert embeddings.shape == (len(chunk_records), 384)
assert np.isfinite(embeddings).all()

print("Embedding validation passed.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embedding shape: (75, 384)
Embedding dtype: float32
Embedding validation passed.


In [18]:
import os
import json
import psycopg
from psycopg.types.json import Jsonb
from pgvector.psycopg import register_vector
from dotenv import load_dotenv

load_dotenv("/content/jev-rag/.env", override=True)
DATABASE_URL = os.environ["DATABASE_URL"]

assert len(chunk_records) == len(embeddings), (
    "Chunk count and embedding count do not match."
)

rows = []

for chunk, embedding in zip(chunk_records, embeddings):
    rows.append((
        chunk["document_id"],
        chunk["title"],
        chunk.get("section"),
        chunk["source_url"],
        chunk["content"],
        chunk["content_hash"],
        Jsonb(chunk["metadata"]),
        embedding.tolist(),
    ))

insert_sql = """
INSERT INTO document_chunks (
    document_id, title, section, source_url,
    content, content_hash, metadata, embedding
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
ON CONFLICT (document_id, content_hash) DO NOTHING;
"""

with psycopg.connect(DATABASE_URL) as conn:
    register_vector(conn)

    with conn.cursor() as cur:
        cur.executemany(insert_sql, rows)
        print("Insert command completed.")

    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM document_chunks;")
        total = cur.fetchone()[0]

        cur.execute("""
            SELECT COUNT(DISTINCT document_id)
            FROM document_chunks;
        """)
        document_count = cur.fetchone()[0]

print("Total chunks stored in database:", total)
print("Distinct source documents:", document_count)

Insert command completed.
Total chunks stored in database: 75
Distinct source documents: 6


In [19]:
with psycopg.connect(DATABASE_URL) as conn:
    register_vector(conn)

    with conn.cursor() as cur:
        cur.execute("""
            SELECT chunk_id, title, section,
                   length(content), embedding IS NOT NULL
            FROM document_chunks
            ORDER BY chunk_id
            LIMIT 5;
        """)

        for row in cur.fetchall():
            print(row)

        cur.execute("""
            SELECT COUNT(*)
            FROM document_chunks
            WHERE embedding IS NULL;
        """)
        missing_embeddings = cur.fetchone()[0]

print("Chunks missing embeddings:", missing_embeddings)
assert missing_embeddings == 0, "Some chunks have no embeddings."
print("Database verification passed.")

(1, 'fastapi.tiangolo.com', 'chunk_1', 743, True)
(2, 'fastapi.tiangolo.com', 'chunk_2', 641, True)
(3, 'fastapi.tiangolo.com', 'chunk_3', 776, True)
(4, 'fastapi.tiangolo.com', 'chunk_4', 795, True)
(5, 'fastapi.tiangolo.com', 'chunk_5', 799, True)
Chunks missing embeddings: 0
Database verification passed.


## 6. Retrieval functions
The three methods below share the canonical chunks stored in Neon: dense vector search, BM25 keyword search, and Reciprocal Rank Fusion (RRF).

In [20]:
import os
import psycopg
from dotenv import load_dotenv
from pgvector.psycopg import register_vector

load_dotenv("/content/jev-rag/.env", override=True)
DATABASE_URL = os.environ["DATABASE_URL"]

def dense_search(query: str, k: int = 5) -> list[dict]:
    if not query.strip():
        raise ValueError("Query cannot be empty.")
    if k < 1:
        raise ValueError("k must be at least 1.")

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )[0].tolist()

    with psycopg.connect(DATABASE_URL) as conn:
        register_vector(conn)

        with conn.cursor() as cur:
            cur.execute(
                """
                SELECT
                    chunk_id,
                    title,
                    section,
                    source_url,
                    content,
                    embedding <=> %s::vector AS distance
                FROM document_chunks
                WHERE embedding IS NOT NULL
                ORDER BY embedding <=> %s::vector
                LIMIT %s;
                """,
                (query_embedding, query_embedding, k),
            )

            rows = cur.fetchall()

    return [
        {
            "chunk_id": row[0],
            "title": row[1],
            "section": row[2],
            "source_url": row[3],
            "content": row[4],
            "distance": float(row[5]),
            "similarity": 1.0 - float(row[5]),
        }
        for row in rows
    ]

print("Dense retriever initialized.")

Dense retriever initialized.


In [21]:
import os
import psycopg
from dotenv import load_dotenv

load_dotenv("/content/jev-rag/.env", override=True)
DATABASE_URL = os.environ["DATABASE_URL"]

with psycopg.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT chunk_id, title, section, source_url, content
            FROM document_chunks
            ORDER BY chunk_id;
        """)
        db_chunks = cur.fetchall()

bm25_documents = [
    {
        "chunk_id": row[0],
        "title": row[1],
        "section": row[2],
        "source_url": row[3],
        "content": row[4],
    }
    for row in db_chunks
]

print("Chunks loaded from Neon:", len(bm25_documents))
assert bm25_documents, "No chunks found in Neon."

Chunks loaded from Neon: 75


In [22]:
import re
from rank_bm25 import BM25Okapi

STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been",
    "to", "of", "in", "on", "for", "and", "or", "with", "by",
    "how", "do", "does", "i", "you", "your", "it", "this",
    "that", "can", "could", "would", "should", "from", "as",
}

def tokenize(text: str) -> list[str]:
    text = text.lower()

    # Preserve identifiers and technical tokens like item_id, @app.get.
    tokens = re.findall(
        r"@[a-z_][a-z0-9_.]*|[a-z_][a-z0-9_]*|[0-9]+",
        text,
    )

    return [token for token in tokens if token not in STOPWORDS]

bm25_corpus = [
    tokenize(doc["title"] + " " + doc["content"])
    for doc in bm25_documents
]

bm25_index = BM25Okapi(bm25_corpus)

print("BM25 index rebuilt.")
print("Indexed chunks:", len(bm25_corpus))

BM25 index rebuilt.
Indexed chunks: 75


In [23]:
def bm25_search(query: str, k: int = 5) -> list[dict]:
    if not query.strip():
        raise ValueError("Query cannot be empty.")

    query_tokens = tokenize(query)
    scores = bm25_index.get_scores(query_tokens)

    ranked_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True,
    )

    results = []

    for idx in ranked_indices:
        if scores[idx] <= 0:
            continue

        doc = bm25_documents[idx]
        results.append({
            **doc,
            "bm25_score": float(scores[idx]),
        })

        if len(results) >= k:
            break

    return results

print("BM25 retriever initialized.")

BM25 retriever initialized.


In [24]:
def reciprocal_rank_fusion(
    dense_results: list[dict],
    bm25_results: list[dict],
    k: int = 60,
    top_n: int = 5,
) -> list[dict]:
    """
    Combine dense and BM25 rankings using Reciprocal Rank Fusion.
    RRF score = sum(1 / (k + rank)).
    """
    fused = {}

    for rank, result in enumerate(dense_results, start=1):
        chunk_id = result["chunk_id"]
        fused.setdefault(chunk_id, {
            **result,
            "rrf_score": 0.0,
            "dense_rank": None,
            "bm25_rank": None,
        })
        fused[chunk_id]["rrf_score"] += 1 / (k + rank)
        fused[chunk_id]["dense_rank"] = rank

    for rank, result in enumerate(bm25_results, start=1):
        chunk_id = result["chunk_id"]
        fused.setdefault(chunk_id, {
            **result,
            "rrf_score": 0.0,
            "dense_rank": None,
            "bm25_rank": None,
        })
        fused[chunk_id]["rrf_score"] += 1 / (k + rank)
        fused[chunk_id]["bm25_rank"] = rank

    ranked = sorted(
        fused.values(),
        key=lambda item: item["rrf_score"],
        reverse=True,
    )

    return ranked[:top_n]

## 7. Compare retrieval methods
Change `query` to test another question. This is the single example cell for inspecting all three methods.

In [25]:
query = "How do I define a path parameter in FastAPI?"

dense_results = dense_search(query, k=5)
bm25_results = bm25_search(query, k=5)

hybrid_results = reciprocal_rank_fusion(
    dense_search(query, k=10),
    bm25_search(query, k=10),
    k=60,
    top_n=5,
)

for approach, results in [
    ("Dense", dense_results),
    ("BM25", bm25_results),
    ("Hybrid RRF", hybrid_results),
]:
    print(f"\\n{'=' * 80}\\n{approach}")
    for rank, result in enumerate(results, start=1):
        print(
            f"\\n{rank}. Chunk {result['chunk_id']} | "
            f"{result.get('title', '')} | {result.get('section', '')}"
        )
        score_key = {
            "Dense": "similarity",
            "BM25": "bm25_score",
            "Hybrid RRF": "rrf_score",
        }[approach]
        if score_key in result:
            print(f"{score_key}: {result[score_key]:.4f}")
        print(result["content"][:450].replace("\\n", " "))


\n================================================================================\nDense
\n1. Chunk 55 | path-params | chunk_13
similarity: 0.8458
But you need file_path itself to contain a path, like home/johndoe/myfile.txt.
So, the URL for that file would be something like: /files/home/johndoe/myfile.txt.
OpenAPI support¶
OpenAPI doesn't support a way to declare a path parameter to contain a path inside, as that could lead to scenarios that are difficult to test and define.
Nevertheless, you can still do it in FastAPI, using one of the internal tools from Starlette.
And the docs would sti
\n2. Chunk 43 | path-params | chunk_1
similarity: 0.8385
Path Parameters¶
You can declare path "parameters" or "variables" with the same syntax used by Python format strings:
from fastapi import FastAPI
app = FastAPI()
@app.get("/items/{item_id}")
async def read_item(item_id):
return {"item_id": item_id}
The value of the path parameter item_id will be passed to your function as the argument item_id

## 8. Retrieval audit

Eight evaluation questions are prepared, but the relevant chunk IDs must be confirmed from the actual chunk contents before metrics are trusted.

**Next step:** build one relevance-review table with one row per unique `(question, chunk_id)` pair, label each pair `1` (relevant) or `0` (not relevant), then calculate Recall@5, MRR@5, and nDCG@5. The previous notebook contained multiple overlapping audit/label cells; those have been removed from this clean version to avoid conflicting labels and duplicate code.

The current candidate IDs are not treated as final labels in this notebook.


In [32]:

import pandas as pd

# 1. Evaluation questions
golden_dataset = [
    {"id": "Q1", "question": "What is FastAPI used for?"},
    {"id": "Q2", "question": "How do I create a basic FastAPI application?"},
    {"id": "Q3", "question": "What is Swagger UI, and how do I access it?"},
    {"id": "Q4", "question": "What is OpenAPI, and where can I view its JSON schema?"},
    {"id": "Q5", "question": "How do I define a GET endpoint?"},
    {"id": "Q6", "question": "How do I define a path parameter?"},
    {"id": "Q7", "question": "How do I declare query parameters?"},
    {"id": "Q8", "question": "How do I define a request body using Pydantic?"},
]

# 2. Run each retrieval method and collect ranked results
audit_rows = []

for item in golden_dataset:
    question = item["question"]

    dense_results = dense_search(question, k=5)
    bm25_results = bm25_search(question, k=5)

    hybrid_results = reciprocal_rank_fusion(
        dense_search(question, k=10),
        bm25_search(question, k=10),
        k=60,
        top_n=5,
    )

    approaches = [
        ("Dense", dense_results),
        ("BM25", bm25_results),
        ("Hybrid RRF", hybrid_results),
    ]

    for approach, results in approaches:
        for rank, result in enumerate(results, start=1):
            audit_rows.append({
                "Question ID": item["id"],
                "Question": question,
                "Approach": approach,
                "Rank": rank,
                "Chunk ID": result["chunk_id"],
                "Title": result.get("title", ""),
                "Section": result.get("section", ""),
                "Content Preview": (
                    result.get("content", "")
                    .replace("\n", " ")[:500]
                ),
            })

# 3. Store the complete retrieval audit
audit_df = pd.DataFrame(audit_rows)

# 4. Create one review row per question–chunk pair
review_columns = [
    "Question ID",
    "Question",
    "Chunk ID",
    "Title",
    "Section",
    "Content Preview",
]

unique_review_df = (
    audit_df[review_columns]
    .drop_duplicates(subset=["Question ID", "Chunk ID"])
    .sort_values(["Question ID", "Chunk ID"])
    .reset_index(drop=True)
)

# All labels start blank for genuine manual review
label_col = "Relevant (1=yes, 0=no)"
unique_review_df[label_col] = pd.NA

# 5. Export the review file
review_path = "retrieval_relevance_review.csv"
unique_review_df.to_csv(review_path, index=False)

# 6. Report checks
print("Retrieval audit rows:", len(audit_df))
print("Unique questions:", audit_df["Question ID"].nunique())
print("Unique question–chunk pairs:", len(unique_review_df))
print("Duplicate pairs removed:", len(audit_df) - len(unique_review_df))
print("Unlabelled pairs:", unique_review_df[label_col].isna().sum())

print("\nResults per question and approach:")
print(
    audit_df.groupby(["Question ID", "Approach"])
    .size()
    .unstack(fill_value=0)
)

print(f"\nReview file saved: {review_path}")
display(unique_review_df)

Retrieval audit rows: 120
Unique questions: 8
Unique question–chunk pairs: 65
Duplicate pairs removed: 55
Unlabelled pairs: 65

Results per question and approach:
Approach     BM25  Dense  Hybrid RRF
Question ID                         
Q1              5      5           5
Q2              5      5           5
Q3              5      5           5
Q4              5      5           5
Q5              5      5           5
Q6              5      5           5
Q7              5      5           5
Q8              5      5           5

Review file saved: retrieval_relevance_review.csv


,Question ID,Question,Chunk ID,Title,Section,Content Preview,"Relevant (1=yes, 0=no)"
0,Q1,What is FastAPI used for?,1,fastapi.tiangolo.com,chunk_1,"FastAPI Conf '26 October 28, 2026 Amsterdam, N...",<NA>
1,Q1,What is FastAPI used for?,3,fastapi.tiangolo.com,chunk_3,Sponsors¶ Keystone Sponsor¶ Gold Sponsors¶ Sil...,<NA>
2,Q1,What is FastAPI used for?,4,fastapi.tiangolo.com,chunk_4,"(ref) ""[...] I'm using FastAPI a ton these day...",<NA>
3,Q1,What is FastAPI used for?,15,fastapi.tiangolo.com,chunk_15,- A very powerful and easy to use Dependency I...,<NA>
4,Q1,What is FastAPI used for?,17,fastapi.tiangolo.com,chunk_17,Deploy to other cloud providers¶ FastAPI is op...,<NA>
...,...,...,...,...,...,...,...
60,Q8,How do I define a request body using Pydantic?,39,first-steps,chunk_12,FastAPI doesn't enforce any specific meaning. ...,<NA>
61,Q8,How do I define a request body using Pydantic?,65,body,chunk_1,Request Body¶ When you need to send data from ...,<NA>
62,Q8,How do I define a request body using Pydantic?,68,body,chunk_4,"To add it to your path operation, declare it t...",<NA>
63,Q8,How do I define a request body using Pydantic?,72,body,chunk_8,price_with_tax = item.price + item.tax item_di...,<NA>


In [37]:

import os
import getpass
import requests
import pandas as pd

api_key = getpass.getpass("Enter your NEW OpenRouter API key: ")

review_df = pd.read_csv("retrieval_relevance_review.csv")
label_col = "Relevant (1=yes, 0=no)"
review_df[label_col] = pd.to_numeric(review_df[label_col], errors="coerce")

pending = review_df[review_df[label_col].isna()]
if pending.empty:
    raise ValueError("No unlabelled rows remain.")

row = pending.iloc[0]

payload = {
    "model": "typesafe/jev-1.13",
    "state": (
        f"Question: {row['Question']}\n\n"
        f"Retrieved chunk:\n{row['Content Preview']}"
    ),
    "questions": {
        "relevant": {
            "type": "noul",
            "instructions": (
                "Does this retrieved chunk provide useful evidence "
                "for answering the question? Answer yes if it helps "
                "answer the question, even partially. Answer no if it "
                "is unrelated or only shares keywords."
            ),
        }
    },
}

response = requests.post(
    "https://openrouter.ai/api/alpha/decisions",
    headers={
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    },
    json=payload,
    timeout=60,
)

print("HTTP status:", response.status_code)
print(response.text[:4000])
response.raise_for_status()


Enter your NEW OpenRouter API key: ··········
HTTP status: 200
{"model":"typesafe/jev-1.13-20260917","answers":{"relevant":{"type":"noul","noul":0.93}},"usage":{"input_tokens":432,"output_tokens":22,"cost":0.000018144},"id":"gen-dec-1791740553-ekVjv8RiIXZIUhGIXYCf","provider":"TypeSafe"}


In [38]:

import os
import json
import time
import pandas as pd
import requests
from getpass import getpass

# 1. Load the review CSV
review_path = "retrieval_relevance_review.csv"
review_df = pd.read_csv(review_path)

label_col = "Relevant (1=yes, 0=no)"
score_col = "JEV Relevance Score"

required_cols = [
    "Question ID",
    "Question",
    "Chunk ID",
    "Content Preview",
    label_col,
]

missing_cols = [c for c in required_cols if c not in review_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

review_df[label_col] = pd.to_numeric(
    review_df[label_col], errors="coerce"
)

if score_col not in review_df.columns:
    review_df[score_col] = pd.NA

if "JEV Reason" not in review_df.columns:
    review_df["JEV Reason"] = ""

# Process only rows that have neither a label nor a saved score.
pending_indices = review_df.index[
    review_df[label_col].isna()
    & review_df[score_col].isna()
].tolist()

if not pending_indices:
    print("No unprocessed question–chunk pairs found.")
else:
    api_key = getpass("Enter your NEW OpenRouter API key: ")

    endpoint = "https://openrouter.ai/api/alpha/decisions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    model = "typesafe/jev-1.13"
    batch_size = 5

    print("Unlabelled pairs to judge:", len(pending_indices))

    # Keep the same relevance criteria from the previous prompt.
    instructions = """
You are evaluating retrieval relevance for a RAG system using official
FastAPI documentation.

Judge whether the supplied chunk contains information that helps answer
the question.

Rules:
- Relevant = 1 if the chunk provides useful evidence for answering the question.
- Relevant = 0 if it is unrelated, only shares keywords, or does not help answer it.
- Do not assume information that is absent from the chunk.
- Judge the chunk itself, not its rank or retrieval method.
- A chunk can be relevant even if it does not provide the complete answer.
"""

    for start in range(0, len(pending_indices), batch_size):
        batch_indices = pending_indices[start:start + batch_size]

        try:
            for row_idx in batch_indices:
                row = review_df.loc[row_idx]

                state = f"""
Question ID: {row['Question ID']}

Question:
{row['Question']}

Retrieved chunk:
{row['Content Preview']}
"""

                payload = {
                    "model": model,
                    "state": state,
                    "questions": {
                        "relevant": {
                            "type": "noul",
                            "instructions": (
                                instructions
                                + """
Return a relevance judgment for this question–chunk pair.
Use the model's numeric NOUL score as the result.
Do not judge based on retrieval rank or method.
"""
                            ),
                        }
                    },
                }

                response = requests.post(
                    endpoint,
                    headers=headers,
                    json=payload,
                    timeout=60,
                )
                response.raise_for_status()

                result = response.json()
                answer = result["answers"]["relevant"]
                score = answer.get("noul")

                if not isinstance(score, (int, float)):
                    raise ValueError(
                        f"Unexpected NOUL response for row {row_idx}: {answer}"
                    )

                # Save the raw score; do not assume a threshold.
                review_df.at[row_idx, score_col] = float(score)
                review_df.at[row_idx, "JEV Reason"] = (
                    "Raw NOUL score recorded; binary label not assigned."
                )

            # Save after every completed batch.
            review_df.to_csv(review_path, index=False)

            completed = min(start + len(batch_indices), len(pending_indices))
            print(f"Scored {completed}/{len(pending_indices)} pairs")

        except Exception as exc:
            review_df.to_csv(review_path, index=False)
            print("Batch failed. Completed scores were saved.")
            print("Error:", exc)
            raise

        if start + batch_size < len(pending_indices):
            time.sleep(1)

    print("\nJEV scoring completed.")
    print("Total rows:", len(review_df))
    print("Scores recorded:", review_df[score_col].notna().sum())
    print("Labels still blank:", review_df[label_col].isna().sum())
    print("Updated file:", review_path)

    display(
        review_df[
            [
                "Question ID",
                "Question",
                "Chunk ID",
                score_col,
                label_col,
                "JEV Reason",
            ]
        ].head(15)
    )

Enter your NEW OpenRouter API key: ··········
Unlabelled pairs to judge: 64


/tmp/ipykernel_1175/2298013310.py:129: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Raw NOUL score recorded; binary label not assigned.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  review_df.at[row_idx, "JEV Reason"] = (


Scored 5/64 pairs
Scored 10/64 pairs
Scored 15/64 pairs
Scored 20/64 pairs
Scored 25/64 pairs
Scored 30/64 pairs
Scored 35/64 pairs
Scored 40/64 pairs
Scored 45/64 pairs
Scored 50/64 pairs
Scored 55/64 pairs
Scored 60/64 pairs
Scored 64/64 pairs

JEV scoring completed.
Total rows: 65
Scores recorded: 64
Labels still blank: 64
Updated file: retrieval_relevance_review.csv


,Question ID,Question,Chunk ID,JEV Relevance Score,"Relevant (1=yes, 0=no)",JEV Reason
0,Q1,What is FastAPI used for?,1,<NA>,0.0,NaN
1,Q1,What is FastAPI used for?,3,0.84,NaN,Raw NOUL score recorded; binary label not assi...
2,Q1,What is FastAPI used for?,4,0.85,NaN,Raw NOUL score recorded; binary label not assi...
3,Q1,What is FastAPI used for?,15,0.87,NaN,Raw NOUL score recorded; binary label not assi...
4,Q1,What is FastAPI used for?,17,0.38,NaN,Raw NOUL score recorded; binary label not assi...
5,Q1,What is FastAPI used for?,18,0.37,NaN,Raw NOUL score recorded; binary label not assi...
6,Q1,What is FastAPI used for?,21,0.38,NaN,Raw NOUL score recorded; binary label not assi...
7,Q1,What is FastAPI used for?,23,0.21,NaN,Raw NOUL score recorded; binary label not assi...
8,Q1,What is FastAPI used for?,32,0.33,NaN,Raw NOUL score recorded; binary label not assi...
9,Q1,What is FastAPI used for?,35,0.41,NaN,Raw NOUL score recorded; binary label not assi...


In [39]:

import pandas as pd

review_path = "retrieval_relevance_review.csv"
df = pd.read_csv(review_path)

score_col = "JEV Relevance Score"
label_col = "Relevant (1=yes, 0=no)"

df[score_col] = pd.to_numeric(df[score_col], errors="coerce")
df[label_col] = pd.to_numeric(df[label_col], errors="coerce")

print("Score summary:")
display(df[score_col].describe())

print("\nScores by question:")
display(
    df.groupby("Question ID")[score_col]
    .agg(["count", "min", "mean", "max"])
    .round(3)
)

print("\nExisting binary labels:")
print(df[label_col].value_counts(dropna=False))

print("\nLowest scores:")
display(
    df.nsmallest(10, score_col)[
        ["Question ID", "Question", "Chunk ID", score_col, label_col]
    ]
)

print("\nHighest scores:")
display(
    df.nlargest(10, score_col)[
        ["Question ID", "Question", "Chunk ID", score_col, label_col]
    ]
)


Score summary:


,JEV Relevance Score
count,64.000000
mean,0.614531
std,0.325883
min,0.050000
25%,0.327500
50%,0.745000
75%,0.902500
max,0.970000



Scores by question:


,count,min,mean,max
Question ID,,,,
Q1,9,0.21,0.516,0.87
Q2,12,0.10,0.646,0.97
Q3,6,0.05,0.445,0.86
Q4,6,0.12,0.508,0.82
Q5,9,0.15,0.710,0.97
Q6,10,0.06,0.574,0.97
Q7,6,0.57,0.782,0.97
Q8,6,0.05,0.733,0.97



Existing binary labels:
Relevant (1=yes, 0=no)
NaN    64
0.0     1
Name: count, dtype: int64

Lowest scores:


,Question ID,Question,Chunk ID,JEV Relevance Score,"Relevant (1=yes, 0=no)"
26,Q3,"What is Swagger UI, and how do I access it?",45,0.05,NaN
60,Q8,How do I define a request body using Pydantic?,39,0.05,NaN
43,Q6,How do I define a path parameter?,37,0.06,NaN
45,Q6,How do I define a path parameter?,42,0.09,NaN
18,Q2,How do I create a basic FastAPI application?,32,0.10,NaN
44,Q6,How do I define a path parameter?,39,0.11,NaN
12,Q2,How do I create a basic FastAPI application?,15,0.12,NaN
32,Q4,"What is OpenAPI, and where can I view its JSON...",68,0.12,NaN
40,Q5,How do I define a GET endpoint?,55,0.15,NaN
42,Q5,How do I define a GET endpoint?,68,0.18,NaN



Highest scores:


,Question ID,Question,Chunk ID,JEV Relevance Score,"Relevant (1=yes, 0=no)"
10,Q2,How do I create a basic FastAPI application?,6,0.97,NaN
17,Q2,How do I create a basic FastAPI application?,28,0.97,NaN
21,Q2,How do I create a basic FastAPI application?,42,0.97,NaN
37,Q5,How do I define a GET endpoint?,40,0.97,NaN
46,Q6,How do I define a path parameter?,43,0.97,NaN
49,Q6,How do I define a path parameter?,56,0.97,NaN
54,Q7,How do I declare query parameters?,58,0.97,NaN
62,Q8,How do I define a request body using Pydantic?,68,0.97,NaN
20,Q2,How do I create a basic FastAPI application?,36,0.96,NaN
34,Q5,How do I define a GET endpoint?,36,0.96,NaN


In [40]:

import pandas as pd

review_path = "retrieval_relevance_review.csv"
df = pd.read_csv(review_path)

score_col = "JEV Relevance Score"
label_col = "Relevant (1=yes, 0=no)"

df[score_col] = pd.to_numeric(df[score_col], errors="coerce")
df[label_col] = pd.to_numeric(df[label_col], errors="coerce")

# Ensure text columns have the correct dtype.
df["JEV Reason"] = df["JEV Reason"].astype("string")

show_cols = [
    "Question ID",
    "Question",
    "Chunk ID",
    "Content Preview",
    score_col,
    label_col,
]

print("LOWEST SCORES — inspect for irrelevant chunks")
display(
    df.dropna(subset=[score_col])
      .nsmallest(8, score_col)[show_cols]
)

print("HIGHEST SCORES — inspect for relevant chunks")
display(
    df.dropna(subset=[score_col])
      .nlargest(8, score_col)[show_cols]
)

LOWEST SCORES — inspect for irrelevant chunks


,Question ID,Question,Chunk ID,Content Preview,JEV Relevance Score,"Relevant (1=yes, 0=no)"
26,Q3,"What is Swagger UI, and how do I access it?",45,"""path"", ""item_id"" ], ""msg"": ""Input should be a...",0.05,NaN
60,Q8,How do I define a request body using Pydantic?,39,FastAPI doesn't enforce any specific meaning. ...,0.05,NaN
43,Q6,How do I define a path parameter?,37,One of: - POST - GET - PUT - DELETE ...and the...,0.06,NaN
45,Q6,How do I define a path parameter?,42,Recap¶ - Import FastAPI. - Create an app insta...,0.09,NaN
18,Q2,How do I create a basic FastAPI application?,32,You can see it directly at: It will show a JSO...,0.10,NaN
44,Q6,How do I define a path parameter?,39,FastAPI doesn't enforce any specific meaning. ...,0.11,NaN
12,Q2,How do I create a basic FastAPI application?,15,- A very powerful and easy to use Dependency I...,0.12,NaN
32,Q4,"What is OpenAPI, and where can I view its JSON...",68,"To add it to your path operation, declare it t...",0.12,NaN


HIGHEST SCORES — inspect for relevant chunks


,Question ID,Question,Chunk ID,Content Preview,JEV Relevance Score,"Relevant (1=yes, 0=no)"
10,Q2,How do I create a basic FastAPI application?,6,Example¶ Create it¶ Create a file main.py with...,0.97,NaN
17,Q2,How do I create a basic FastAPI application?,28,First Steps¶ The simplest FastAPI file could l...,0.97,NaN
21,Q2,How do I create a basic FastAPI application?,42,Recap¶ - Import FastAPI. - Create an app insta...,0.97,NaN
37,Q5,How do I define a GET endpoint?,40,You could also define it as a normal function ...,0.97,NaN
46,Q6,How do I define a path parameter?,43,"Path Parameters¶ You can declare path ""paramet...",0.97,NaN
49,Q6,How do I define a path parameter?,56,"/files/{file_path:path} In this case, the name...",0.97,NaN
54,Q7,How do I declare query parameters?,58,Query Parameters¶ When you declare other funct...,0.97,NaN
62,Q8,How do I define a request body using Pydantic?,68,"To add it to your path operation, declare it t...",0.97,NaN


In [41]:
import pandas as pd

review_path = "retrieval_relevance_review.csv"
df = pd.read_csv(review_path)

score_col = "JEV Relevance Score"
label_col = "Relevant (1=yes, 0=no)"

df[score_col] = pd.to_numeric(df[score_col], errors="coerce")
df[label_col] = pd.to_numeric(df[label_col], errors="coerce")

threshold = 0.70
pending = df[label_col].isna() & df[score_col].notna()

df.loc[pending, label_col] = (
    df.loc[pending, score_col] >= threshold
).astype(int)

df.to_csv(review_path, index=False)

print("Labels assigned:", int(pending.sum()))
print("Labels still blank:", int(df[label_col].isna().sum()))
print("\nRelevance label counts:")
print(df[label_col].value_counts().sort_index())

Labels assigned: 64
Labels still blank: 0

Relevance label counts:
Relevant (1=yes, 0=no)
0.0    30
1.0    35
Name: count, dtype: int64


## 9. Complete relevance review and calculate metrics

1. Run Section 8 to create `retrieval_relevance_review.csv`.
2. Open the CSV from Colab's Files panel, download it, and label each row in `Relevant (1=yes, 0=no)`:
   - `1` = the chunk contains information that helps answer that question.
   - `0` = the chunk does not help answer that question.
3. Upload the completed CSV back to the Colab session, replacing the exported file.
4. Run the next cell. It will stop if any labels are blank or invalid.

Each question–chunk pair appears once in the review file, even if multiple retrievers returned it. The same label is then used for Dense, BM25, and Hybrid RRF.


In [42]:
from pathlib import Path
import math
import pandas as pd

REVIEW_PATH = Path("retrieval_relevance_review.csv")
if not REVIEW_PATH.exists():
    # Colab upload dialog appears only if the reviewed CSV is not already present.
    from google.colab import files
    uploaded = files.upload()
    uploaded_csv = next(
        (name for name in uploaded if name.endswith("retrieval_relevance_review.csv")),
        None,
    )
    assert uploaded_csv, "Upload the completed retrieval_relevance_review.csv file."
    REVIEW_PATH = Path(uploaded_csv)

labels_df = pd.read_csv(REVIEW_PATH)
label_col = "Relevant (1=yes, 0=no)"
required_cols = {"Question ID", "Chunk ID", label_col}
assert required_cols.issubset(labels_df.columns), (
    f"Review file must contain columns: {sorted(required_cols)}"
)

labels_df[label_col] = pd.to_numeric(labels_df[label_col], errors="coerce")
assert labels_df[label_col].notna().all(), "Some relevance labels are blank or invalid."
assert labels_df[label_col].isin([0, 1]).all(), "Labels must be 0 or 1."
assert not labels_df.duplicated(["Question ID", "Chunk ID"]).any(), (
    "Each question–chunk pair must appear only once."
)

# Use the exact retrieval rankings generated in Section 8.
evaluated = audit_df.merge(
    labels_df[["Question ID", "Chunk ID", label_col]],
    on=["Question ID", "Chunk ID"],
    how="left",
    validate="many_to_one",
)
assert evaluated[label_col].notna().all(), (
    "Some retrieved question–chunk pairs are missing labels. "
    "Re-run Section 8 and label every row in the exported CSV."
)

# The review candidates are the union of the top-5 results from each method.
# Recall is therefore measured against the relevant chunks found in this reviewed pool.
def _metrics_at_k(retrieved_ids, relevant_ids, k=5):
    retrieved = retrieved_ids[:k]
    relevant = set(relevant_ids)
    if not relevant:
        return {"Recall@5": float("nan"), "MRR@5": 0.0, "nDCG@5": 0.0}

    hits = [1 if chunk_id in relevant else 0 for chunk_id in retrieved]
    recall = sum(hits) / len(relevant)
    reciprocal_rank = next(
        (1 / rank for rank, hit in enumerate(hits, start=1) if hit), 0.0
    )
    dcg = sum(hit / math.log2(rank + 1) for rank, hit in enumerate(hits, start=1))
    ideal_hits = min(len(relevant), k)
    idcg = sum(1 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return {
        "Recall@5": recall,
        "MRR@5": reciprocal_rank,
        "nDCG@5": dcg / idcg if idcg else 0.0,
    }

relevant_by_question = (
    labels_df[labels_df[label_col] == 1]
    .groupby("Question ID")["Chunk ID"]
    .apply(set)
    .to_dict()
)

metric_rows = []
for (question_id, question, approach), group in evaluated.groupby(
    ["Question ID", "Question", "Approach"], sort=False
):
    group = group.sort_values("Rank")
    scores = _metrics_at_k(
        group["Chunk ID"].tolist(),
        relevant_by_question.get(question_id, set()),
        k=5,
    )
    metric_rows.append({
        "Question ID": question_id,
        "Question": question,
        "Approach": approach,
        **scores,
    })

per_question_metrics = pd.DataFrame(metric_rows)
summary_metrics = (
    per_question_metrics.groupby("Approach")[["Recall@5", "MRR@5", "nDCG@5"]]
    .mean()
    .sort_values("nDCG@5", ascending=False)
    .reset_index()
)

print("Per-question retrieval metrics:")
print(per_question_metrics.round(3).to_string(index=False))
print("\nAverage retrieval metrics:")
print(summary_metrics.round(3).to_string(index=False))

Per-question retrieval metrics:
Question ID                                               Question   Approach  Recall@5  MRR@5  nDCG@5
         Q1                              What is FastAPI used for?      Dense     1.000   1.00   0.853
         Q1                              What is FastAPI used for?       BM25     0.000   0.00   0.000
         Q1                              What is FastAPI used for? Hybrid RRF     0.333   0.25   0.202
         Q2           How do I create a basic FastAPI application?      Dense     0.571   1.00   0.830
         Q2           How do I create a basic FastAPI application?       BM25     0.286   1.00   0.509
         Q2           How do I create a basic FastAPI application? Hybrid RRF     0.429   1.00   0.699
         Q3            What is Swagger UI, and how do I access it?      Dense     1.000   1.00   0.920
         Q3            What is Swagger UI, and how do I access it?       BM25     1.000   1.00   1.000
         Q3            What is Swagger UI

In [43]:

import pandas as pd
import numpy as np

# Load the reviewed relevance labels
review_df = pd.read_csv("retrieval_relevance_review.csv")

qid = "Question ID"
chunk = "Chunk ID"
label = "Relevant (1=yes, 0=no)"

# 1. Validate relevance labels
review_df[label] = pd.to_numeric(review_df[label], errors="coerce")

duplicate_pairs = review_df.duplicated([qid, chunk]).sum()
missing_labels = review_df[label].isna().sum()
invalid_labels = (~review_df[label].isin([0, 1]) & review_df[label].notna()).sum()

print("LABEL VALIDATION")
print("Duplicate question-chunk pairs:", duplicate_pairs)
print("Missing labels:", missing_labels)
print("Invalid labels:", invalid_labels)

# Stop if labels are not ready for evaluation
if duplicate_pairs or missing_labels or invalid_labels:
    raise ValueError("Fix the relevance labels before evaluating retrieval.")

# 2. Attach labels to the retrieval audit
required = [qid, "Approach", "Rank", chunk]
missing_columns = [c for c in required if c not in audit_df.columns]

if missing_columns:
    raise ValueError(
        f"audit_df is missing columns: {missing_columns}. "
        f"Available columns: {audit_df.columns.tolist()}"
    )

ranked = audit_df.merge(
    review_df[[qid, chunk, label]],
    on=[qid, chunk],
    how="left",
    validate="many_to_one"
)

if ranked[label].isna().any():
    raise ValueError("Some retrieved chunks have no relevance label.")

ranked["Rank"] = pd.to_numeric(ranked["Rank"], errors="raise")
ranked = ranked.sort_values([qid, "Approach", "Rank"])

# One result per question, approach, and chunk
ranked = ranked.drop_duplicates([qid, "Approach", chunk])

# 3. Recalculate Recall@5, MRR@5, and nDCG@5
results = []

for (question_id, approach), group in ranked.groupby([qid, "Approach"]):
    group = group.sort_values("Rank").head(5)
    relevance = group[label].astype(int).tolist()

    # Total relevant chunks in the judged candidate pool
    total_relevant = int(
        review_df.loc[review_df[qid] == question_id, label].sum()
    )

    recall = sum(relevance) / total_relevant if total_relevant else 0.0

    relevant_ranks = [
        rank for rank, rel in enumerate(relevance, start=1) if rel == 1
    ]
    mrr = 1 / relevant_ranks[0] if relevant_ranks else 0.0

    dcg = sum(
        rel / np.log2(rank + 1)
        for rank, rel in enumerate(relevance, start=1)
    )
    ideal_relevance = sorted(
        review_df.loc[review_df[qid] == question_id, label].astype(int),
        reverse=True
    )[:5]
    idcg = sum(
        rel / np.log2(rank + 1)
        for rank, rel in enumerate(ideal_relevance, start=1)
    )
    ndcg = dcg / idcg if idcg else 0.0

    results.append({
        "Question ID": question_id,
        "Approach": approach,
        "Recall@5": recall,
        "MRR@5": mrr,
        "nDCG@5": ndcg
    })

checked_metrics = pd.DataFrame(results)

print("\nRECALCULATED AVERAGE METRICS")
display(
    checked_metrics.groupby("Approach")[
        ["Recall@5", "MRR@5", "nDCG@5"]
    ].mean().round(3).sort_values("Recall@5", ascending=False)
)

print("\nPER-QUESTION METRICS")
display(checked_metrics.round(3))

LABEL VALIDATION
Duplicate question-chunk pairs: 0
Missing labels: 0
Invalid labels: 0

RECALCULATED AVERAGE METRICS


,Recall@5,MRR@5,nDCG@5
Approach,,,
Dense,0.842,1.000,0.897
Hybrid RRF,0.751,0.906,0.805
BM25,0.584,0.812,0.631



PER-QUESTION METRICS


,Question ID,Approach,Recall@5,MRR@5,nDCG@5
0,Q1,BM25,0.000,0.00,0.000
1,Q1,Dense,1.000,1.00,0.853
2,Q1,Hybrid RRF,0.333,0.25,0.202
3,Q2,BM25,0.286,1.00,0.509
4,Q2,Dense,0.571,1.00,0.830
5,Q2,Hybrid RRF,0.429,1.00,0.699
6,Q3,BM25,1.000,1.00,1.000
7,Q3,Dense,1.000,1.00,0.920
8,Q3,Hybrid RRF,1.000,1.00,1.000
9,Q4,BM25,1.000,0.50,0.693


## LLM GENERATOR

In [44]:

import os
import requests
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise ValueError("OPENROUTER_API_KEY is missing from your environment.")

response = requests.get(
    "https://openrouter.ai/api/v1/models",
    headers={"Authorization": f"Bearer {api_key}"},
    timeout=30,
)
response.raise_for_status()

models = response.json()["data"]
qwen_models = [
    {
        "id": model["id"],
        "name": model.get("name", ""),
        "context_length": model.get("context_length"),
    }
    for model in models
    if "qwen" in model["id"].lower()
    and "text" in model.get("architecture", {}).get("output_modalities", [])
]

print(f"Qwen text-generation models found: {len(qwen_models)}")
for model in qwen_models:
    print(model)


Qwen text-generation models found: 42
{'id': 'qwen/qwen3.8-max-prime', 'name': 'Qwen: Qwen3.8 Max Prime', 'context_length': 1000000}
{'id': 'qwen/qwen3.8-omni-flash', 'name': 'Qwen: Qwen3.8 Omni Flash', 'context_length': 1000000}
{'id': 'qwen/qwen3.8-max-0902', 'name': 'Qwen: Qwen3.8 Max (0902)', 'context_length': 1000000}
{'id': 'qwen/qwen3.8-flash', 'name': 'Qwen: Qwen3.8 Flash', 'context_length': 1000000}
{'id': 'qwen/qwen3.8-27b', 'name': 'Qwen: Qwen3.8 27B', 'context_length': 1000000}
{'id': 'qwen/qwen3.8-2.4t-a95b', 'name': 'Qwen: Qwen3.8 2.4T A95B', 'context_length': 1048576}
{'id': 'qwen/qwen3.7-flash', 'name': 'Qwen: Qwen3.7 Flash', 'context_length': 1000000}
{'id': 'qwen/qwen3.7-plus', 'name': 'Qwen: Qwen3.7 Plus', 'context_length': 1000000}
{'id': 'qwen/qwen3.7-max', 'name': 'Qwen: Qwen3.7 Max', 'context_length': 1000000}
{'id': 'qwen/qwen3.5-plus-20260420', 'name': 'Qwen: Qwen3.5 Plus 2026-04-20', 'context_length': 1000000}
{'id': 'qwen/qwen3.6-flash', 'name': 'Qwen: Qwen3.

In [58]:

import os
import requests

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
QWEN_MODEL = "qwen/qwen3.8-2.4t-a95b"


def generate_rag_answer(question, k=5):
    api_key = os.getenv("OPENROUTER_API_KEY")
    if not api_key:
        raise ValueError("OPENROUTER_API_KEY is missing.")

    # Retrieve context
    chunks = dense_search(question, k=k)

    if not chunks:
        return {
            "question": question,
            "answer": "No relevant documentation was found.",
            "sources": [],
        }

    context_parts = []
    sources = []

    for rank, chunk in enumerate(chunks, start=1):
        content = str(chunk.get("content", ""))[:2000]
        title = chunk.get("title") or "Untitled"
        section = chunk.get("section") or ""
        source_url = chunk.get("source_url") or ""

        context_parts.append(
            f"[Source {rank}]\n"
            f"Title: {title}\n"
            f"Section: {section}\n"
            f"Content:\n{content}"
        )

        sources.append({
            "rank": rank,
            "title": title,
            "section": section,
            "source_url": source_url,
        })

    context = "\n\n".join(context_parts)

    # Generate answer
    payload = {
        "model": QWEN_MODEL,
        "messages": [
            {
                "role": "system",
                "content": (
                    "Answer the user's question using only the supplied "
                    "FastAPI documentation. Answer the exact question asked. "
                    "Do not answer a different question. If evidence is "
                    "insufficient, say so. Cite evidence using [Source 1], "
                    "[Source 2], etc."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Question:\n{question}\n\n"
                    f"Retrieved documentation:\n{context}"
                ),
            },
        ],
        "temperature": 0,
        "max_tokens": 800,
    }

    response = requests.post(
        OPENROUTER_URL,
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        json=payload,
        timeout=90,
    )

    if not response.ok:
        print("HTTP status:", response.status_code)
        print("API error:", response.text[:2000])
        response.raise_for_status()

    data = response.json()
    choices = data.get("choices", [])

    if not choices:
        raise ValueError(f"No model choices returned: {data}")

    message = choices[0].get("message", {})
    answer = message.get("content")

    if isinstance(answer, list):
        answer = "\n".join(
            item["text"]
            for item in answer
            if isinstance(item, dict) and item.get("text")
        )

    if not answer or not str(answer).strip():
        print("Model message:", message)
        print("Finish reason:", choices[0].get("finish_reason"))
        raise ValueError("The model returned no readable answer.")

    return {
        "question": question,
        "answer": str(answer).strip(),
        "sources": sources,
        "model": data.get("model", QWEN_MODEL),
        "usage": data.get("usage", {}),
    }


In [59]:

result = generate_rag_answer(
    "How do I define a path parameter in FastAPI?"
)

print("QUESTION:", result["question"])
print("\nANSWER:\n", result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print(
        f"[Source {source['rank']}] "
        f"{source['title']} — {source['section']}"
    )

print("\nUSAGE:", result["usage"])


QUESTION: How do I define a path parameter in FastAPI?

ANSWER:
 To define a path parameter in FastAPI, declare it in the route path using `{parameter_name}` syntax, like a Python format string, and add a function parameter with the same name. FastAPI will pass the value from the URL into that function argument. [Source 2]

Example:

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/items/{item_id}")
async def read_item(item_id):
    return {"item_id": item_id}
```

You can also declare the path parameter’s type using standard Python type annotations:

```python
@app.get("/items/{item_id}")
async def read_item(item_id: int):
    return {"item_id": item_id}
```

In this case, `item_id` is declared as an `int`. [Source 2]

If a function parameter is also declared in the path, FastAPI recognizes it as a path parameter. [Source 5]

SOURCES:
[Source 1] path-params — chunk_13
[Source 2] path-params — chunk_1
[Source 3] path-params — chunk_14
[Source 4] query-params — chunk_4

## JEV Answer Evaluation

In [60]:

import os
import requests

JEV_URL = "https://openrouter.ai/api/alpha/decisions"
JEV_MODEL = "typesafe/jev-1.13"


def evaluate_rag_answer(result, k=5):
    api_key = os.getenv("OPENROUTER_API_KEY")
    if not api_key:
        raise ValueError("OPENROUTER_API_KEY is missing.")

    question = result["question"]
    answer = result["answer"]

    # Retrieve the evidence used for evaluation
    chunks = dense_search(question, k=k)

    evidence_parts = []
    for rank, chunk in enumerate(chunks, start=1):
        evidence_parts.append(
            f"[Source {rank}]\n"
            f"Title: {chunk.get('title', 'Untitled')}\n"
            f"Section: {chunk.get('section', '')}\n"
            f"Content: {str(chunk.get('content', ''))[:1500]}"
        )

    evidence = "\n\n".join(evidence_parts)

    state = (
        f"QUESTION:\n{question}\n\n"
        f"GENERATED ANSWER:\n{answer}\n\n"
        f"RETRIEVED EVIDENCE:\n{evidence}\n\n"
        "Evaluate only the supplied answer and evidence. "
        "Do not use outside knowledge."
    )

    payload = {
        "model": JEV_MODEL,
        "state": state,
        "questions": {
            "answer_relevance": {
                "type": "noul",
                "instructions": (
                    "Is the generated answer responsive to the exact "
                    "question asked and does it answer the user's intent?"
                ),
            },
            "groundedness": {
                "type": "noul",
                "instructions": (
                    "Are the factual claims in the generated answer "
                    "supported by the retrieved evidence? Answer yes only "
                    "if the substantive claims are supported and there "
                    "are no material unsupported claims."
                ),
            },
            "citation_support": {
                "type": "noul",
                "instructions": (
                    "Do the source citations in the generated answer "
                    "refer to supplied sources that actually support "
                    "the claims attached to them? Answer no if citations "
                    "are missing where needed, mismatched, or unsupported."
                ),
            },
        },
    }

    response = requests.post(
        JEV_URL,
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        json=payload,
        timeout=60,
    )

    if not response.ok:
        print("HTTP status:", response.status_code)
        print("API error:", response.text[:2000])
        response.raise_for_status()

    data = response.json()
    answers = data.get("answers", {})

    if not answers:
        raise ValueError(f"No JEV answers returned: {data}")

    return {
        "question": question,
        "answer": answer,
        "scores": {
            name: answers[name].get("noul")
            for name in [
                "answer_relevance",
                "groundedness",
                "citation_support",
            ]
            if name in answers
        },
        "raw_response": data,
    }


In [61]:

evaluation = evaluate_rag_answer(result)

print("QUESTION:", evaluation["question"])
print("\nJEV EVALUATION SCORES:")

for criterion, score in evaluation["scores"].items():
    print(f"{criterion}: {score}")


QUESTION: How do I define a path parameter in FastAPI?

JEV EVALUATION SCORES:
answer_relevance: 0.96
groundedness: 0.94
citation_support: 0.86
